In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, T5ForConditionalGeneration, TrainingArguments

In [2]:
train_data  = pd.read_csv(r"C:\Users\srija\OneDrive\Desktop\machine learning\DEEP LEARNING\datasets\samsum-train.csv")
val_data  = pd.read_csv(r"C:\Users\srija\OneDrive\Desktop\machine learning\DEEP LEARNING\datasets\samsum-validation.csv")

In [3]:
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data  = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [4]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags <p>, <h1>
    text = text.strip().lower()
    return text
    

In [5]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [6]:
# Tokenize data
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [7]:
def tokenize(data): 
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids add to labels
    return inputs

In [ ]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()
# input ids =  dialogue => token_ids
# 1=> End of Sequence, => added padding
# attention mask
# labels - target => summary token

In [ ]:
print(len(train_dataset[0]["input_ids"]))
print(len(train_dataset[0]["labels"]))

# as set before.

512
150


In [12]:
# Working with model
# NLP -> Generation Task
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [13]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")

print("device: ", device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
# Training arguments

args = TrainingArguments(
    output_dir = "./results"
)